# 🤖 Chatbot educativo — **Diez Mandamientos (Éxodo 20)**
Motores de recuperación:
1) **TF‑IDF + cosine** (baseline, interpretable)  
2) **Sentence‑BERT** (semántico, opcional; con *fallback* a TF‑IDF)

**Datos**: `faq_exodo20_es.csv`

## 1) Cargar datos

In [2]:
import pandas as pd
from pathlib import Path

CSV_PATH = Path("faq_exodo20_es.csv")
assert CSV_PATH.exists(), "No se encontró faq_exodo20_es.csv."
faq = pd.read_csv(CSV_PATH)
faq

,id,question,answer,refs
0,general_01,¿Cuáles son los diez mandamientos?,Son mandatos dados por Dios a Israel en Éxodo ...,Éxodo 20:1-17
1,general_02,¿Dónde aparecen los diez mandamientos en la Bi...,Aparecen en Éxodo 20:1-17. También se repiten ...,Éxodo 20:1-17; Deuteronomio 5:6-21
2,general_03,¿Cuál es el contexto de Éxodo 20?,Dios habla desde el monte Sinaí después de lib...,Éxodo 19–20
3,cmd_01_a,¿Qué significa 'No tendrás otros dioses delant...,Prohíbe adorar o rendir culto a cualquier otra...,Éxodo 20:3
4,cmd_02_a,¿Qué prohíbe el segundo mandamiento sobre imág...,Prohíbe hacer y adorar ídolos o imágenes con f...,Éxodo 20:4-6
5,cmd_03_a,¿Qué significa 'No tomarás el nombre de Dios e...,"Prohíbe usar el nombre de Dios con ligereza, f...",Éxodo 20:7
6,cmd_04_a,¿Qué implica 'Acuérdate del sábado para santif...,"Dedicar el día séptimo al descanso y a Dios, c...",Éxodo 20:8-11
7,cmd_05_a,¿Qué significa 'Honra a tu padre y a tu madre'?,"Llama a respetar, cuidar y obedecer a los padr...",Éxodo 20:12
8,cmd_06_a,¿Qué significa 'No matarás'?,Prohíbe quitar la vida injustamente. Afirma el...,Éxodo 20:13
9,cmd_07_a,¿Qué significa 'No cometerás adulterio'?,Prohíbe la infidelidad matrimonial y protege l...,Éxodo 20:14


## 2) TF‑IDF + cosine (baseline)

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1)
X = vectorizer.fit_transform(faq["question"].fillna(""))

def retrieve_tfidf(query, topk=5):
    qv = vectorizer.transform([query])
    sims = cosine_similarity(qv, X)[0]
    idx = np.argsort(sims)[::-1][:topk]
    return [(int(i), float(sims[i])) for i in idx]

retrieve_tfidf("¿Qué significa no robarás?")

[(10, 1.0000000000000002),
 (12, 0.411648907069156),
 (8, 0.411648907069156),
 (9, 0.3266282270947308),
 (11, 0.2928364904234686)]

## 3) Sentence‑BERT (opcional) con *fallback*

In [4]:
model = None
sbert_ready = False
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    sbert_ready = True
except Exception as e:
    print("SBERT no disponible; se usará TF‑IDF. Detalle:", e)

if sbert_ready:
    emb_q = model.encode(faq["question"].tolist(), normalize_embeddings=True)
    def retrieve_sbert(query, topk=5):
        q = model.encode([query], normalize_embeddings=True)[0]
        sims = emb_q @ q
        idx = np.argsort(sims)[::-1][:topk]
        return [(int(i), float(sims[i])) for i in idx]
else:
    def retrieve_sbert(query, topk=5):
        return retrieve_tfidf(query, topk=topk)

retrieve_sbert("¿Qué prohíbe el segundo mandamiento?")

SBERT no disponible; se usará TF‑IDF. Detalle: No module named 'sentence_transformers'


[(4, 0.8121283429968523),
 (15, 0.15867590346564575),
 (16, 0.0757517677727999),
 (14, 0.06125212855254887),
 (5, 0.052641606513477615)]

## 4) Respuesta con umbrales

In [5]:
THRESHOLD_CERT = 0.55
THRESHOLD_MAYBE = 0.40

def responder(query, topk=5, metodo="auto"):
    metodo = metodo.lower()
    if metodo == "tfidf":
        rank = retrieve_tfidf(query, topk=topk)
    elif metodo == "sbert":
        rank = retrieve_sbert(query, topk=topk)
    else:
        rank = retrieve_sbert(query, topk=topk)

    if not rank:
        return {"status":"empty", "message":"Sin resultados", "candidatos":[]}

    best_idx, best_sim = rank[0]
    cand = [{
        "score": float(sim),
        "question": faq.iloc[i]["question"],
        "answer": faq.iloc[i]["answer"],
        "refs": faq.iloc[i]["refs"]
    } for i, sim in rank]

    if best_sim >= THRESHOLD_CERT:
        return {"status":"ok", "match": cand[0], "candidatos": cand}
    elif best_sim >= THRESHOLD_MAYBE:
        return {"status":"maybe", "match": cand[0], "candidatos": cand}
    else:
        return {"status":"uncertain", "message":"No estoy seguro. ¿Te refieres a alguna de estas preguntas?", "candidatos": cand}

responder("¿Cuál es el cuarto mandamiento?")

{'status': 'ok',
 'match': {'score': 0.5622723480289273,
  'question': '¿Cuál es el contexto de Éxodo 20?',
  'answer': 'Dios habla desde el monte Sinaí después de liberar a Israel de Egipto. Da los mandamientos como base del pacto.',
  'refs': 'Éxodo 19–20'},
 'candidatos': [{'score': 0.5622723480289273,
   'question': '¿Cuál es el contexto de Éxodo 20?',
   'answer': 'Dios habla desde el monte Sinaí después de liberar a Israel de Egipto. Da los mandamientos como base del pacto.',
   'refs': 'Éxodo 19–20'},
  {'score': 0.171809650320436,
   'question': '¿Qué promesa acompaña el quinto mandamiento?',
   'answer': 'Promesa de bienestar y larga vida en la tierra al honrar a los padres.',
   'refs': 'Éxodo 20:12'},
  {'score': 0.15746928611981897,
   'question': '¿Qué prohíbe el segundo mandamiento sobre imágenes?',
   'answer': 'Prohíbe hacer y adorar ídolos o imágenes con fines de culto. Señala que la adoración es solo a Dios.',
   'refs': 'Éxodo 20:4-6'},
  {'score': 0.1174474897870186

## 5) Chat CLI (opcional)

In [7]:
def chat():
    print("Chatbot Éxodo 20 — escribe 'salir' para terminar.")
    while True:
        q = input("Tú: ").strip()
        if q.lower() in {"salir","exit","quit"}:
            print("Bot: ¡Hasta luego!")
            break
        r = responder(q)
        if r["status"] == "ok":
            m = r["match"]
            print(f"Bot (confianza alta: {m['score']:.2f}):\n{m['answer']}\nFuente: {m['refs']}\n")
        elif r["status"] == "maybe":
            m = r["match"]
            print(f"Bot (posible: {m['score']:.2f}):\n{m['answer']}\nFuente: {m['refs']}")
            print("También podrías haber preguntado:")
            for s in r["candidatos"][1:3]:
                print(" -", s["question"])
            print()
        else:
            print(r["message"])
            for s in r["candidatos"][:3]:
                print(f" - ({s['score']:.2f}) {s['question']}")
            print()

# Para usarlo localmente:
# chat()

In [10]:
chat()

Chatbot Éxodo 20 — escribe 'salir' para terminar.


Tú:  salir


Bot: ¡Hasta luego!
